# Direct Preference Optimization: Efficient Alignment Without Reward Models

## Learning Objectives
1. Understand the mathematical foundations of DPO
2. Implement DPO loss computation and training
3. Compare DPO efficiency vs RLHF
4. Build and train a preference-based alignment system

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from typing import Tuple, Dict, List
import matplotlib.pyplot as plt

# Device setup for reproducibility
np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Level 1: Basic DPO Loss

Understanding the core DPO loss function that enables preference-based optimization.

In [ ]:
def compute_basic_dpo_loss(
    log_prob_chosen: torch.Tensor,
    log_prob_rejected: torch.Tensor,
    ref_log_prob_chosen: torch.Tensor,
    ref_log_prob_rejected: torch.Tensor,
    beta: float = 0.5
) -> torch.Tensor:
    """
    Compute DPO loss: encourage preference of chosen over rejected.
    
    Loss = -log(sigmoid(beta * (log_ratio_model - log_ratio_ref)))
    
    Args:
        log_prob_chosen: Log probability of chosen response from model
        log_prob_rejected: Log probability of rejected response from model
        ref_log_prob_chosen: Log probability of chosen response from reference
        ref_log_prob_rejected: Log probability of rejected response from reference
        beta: Temperature parameter controlling preference strength
    
    Returns:
        DPO loss (scalar)
    """
    # Compute log probability ratios
    model_log_ratio = log_prob_chosen - log_prob_rejected
    ref_log_ratio = ref_log_prob_chosen - ref_log_prob_rejected
    
    # DPO loss: encourage model ratio to exceed reference ratio
    diff = beta * (model_log_ratio - ref_log_ratio)
    loss = -F.logsigmoid(diff).mean()
    
    return loss

# Example: compute DPO loss for synthetic log probabilities
# Higher probability for chosen means better alignment
batch_size = 4

# Simulate log probabilities (higher = better)
log_prob_chosen = torch.tensor([-2.0, -1.5, -1.2, -2.5], device=device)
log_prob_rejected = torch.tensor([-3.0, -2.5, -2.0, -3.5], device=device)
ref_log_prob_chosen = torch.tensor([-2.2, -1.8, -1.5, -2.8], device=device)
ref_log_prob_rejected = torch.tensor([-3.2, -2.8, -2.2, -3.8], device=device)

# Test with different beta values
betas = [0.1, 0.5, 1.0]
for beta in betas:
    loss = compute_basic_dpo_loss(
        log_prob_chosen, log_prob_rejected,
        ref_log_prob_chosen, ref_log_prob_rejected,
        beta=beta
    )
    print(f'Beta={beta}: DPO Loss = {loss.item():.4f}')

## Level 2: Advanced DPO Training System

Full training system with preference data, batch processing, and convergence tracking.

In [ ]:
class DPOTrainer:
    """
    Training coordinator for Direct Preference Optimization.
    """
    
    def __init__(
        self,
        beta: float = 0.5,
        learning_rate: float = 1e-4,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        """
        Initialize DPO trainer.
        
        Args:
            beta: DPO temperature parameter
            learning_rate: Optimizer learning rate
            device: Device for training
        """
        self.beta = beta
        self.learning_rate = learning_rate
        self.device = device
        self.training_history = {
            'losses': [],
            'accuracy': [],
            'log_ratios': []
        }
    
    def compute_dpo_loss_with_metrics(
        self,
        model_log_probs_chosen: torch.Tensor,
        model_log_probs_rejected: torch.Tensor,
        ref_log_probs_chosen: torch.Tensor,
        ref_log_probs_rejected: torch.Tensor
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        Compute DPO loss and additional metrics.
        
        Args:
            model_log_probs_chosen: Model's log probs for chosen
            model_log_probs_rejected: Model's log probs for rejected
            ref_log_probs_chosen: Reference model's log probs for chosen
            ref_log_probs_rejected: Reference model's log probs for rejected
        
        Returns:
            loss: DPO loss value
            metrics: Dictionary of additional metrics
        """
        # Compute log ratios
        model_log_ratio = model_log_probs_chosen - model_log_probs_rejected
        ref_log_ratio = ref_log_probs_chosen - ref_log_probs_rejected
        
        # DPO loss
        diff = self.beta * (model_log_ratio - ref_log_ratio)
        loss = -F.logsigmoid(diff).mean()
        
        # Metrics
        with torch.no_grad():
            # Accuracy: how often model ratio > ref ratio
            accuracy = (model_log_ratio > ref_log_ratio).float().mean().item()
            
            # Average log ratio difference
            avg_log_ratio_diff = (model_log_ratio - ref_log_ratio).mean().item()
        
        return loss, {
            'loss': loss.item(),
            'accuracy': accuracy,
            'avg_log_ratio_diff': avg_log_ratio_diff
        }
    
    def training_step(
        self,
        batch_log_probs_chosen: torch.Tensor,
        batch_log_probs_rejected: torch.Tensor,
        batch_ref_log_probs_chosen: torch.Tensor,
        batch_ref_log_probs_rejected: torch.Tensor,
        optimizer: torch.optim.Optimizer,
        model: torch.nn.Module
    ) -> Dict[str, float]:
        """
        Single training step.
        
        Args:
            batch_log_probs_chosen: Batch of log probs for chosen
            batch_log_probs_rejected: Batch of log probs for rejected
            batch_ref_log_probs_chosen: Reference log probs for chosen
            batch_ref_log_probs_rejected: Reference log probs for rejected
            optimizer: Torch optimizer
            model: Model being trained
        
        Returns:
            Metrics dictionary
        """
        loss, metrics = self.compute_dpo_loss_with_metrics(
            batch_log_probs_chosen,
            batch_log_probs_rejected,
            batch_ref_log_probs_chosen,
            batch_ref_log_probs_rejected
        )
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        return metrics
    
    def train_epoch(
        self,
        model: torch.nn.Module,
        preference_data: List[Dict],
        batch_size: int = 4,
        num_epochs: int = 3
    ) -> List[Dict]:
        """
        Train for multiple epochs on preference data.
        
        Args:
            model: Model to train
            preference_data: List of preference pairs
            batch_size: Batch size
            num_epochs: Number of training epochs
        
        Returns:
            List of metrics for each step
        """
        optimizer = torch.optim.AdamW(model.parameters(), lr=self.learning_rate)
        model.train()
        
        all_metrics = []
        
        for epoch in range(num_epochs):
            # Shuffle data
            indices = torch.randperm(len(preference_data))
            
            for i in range(0, len(indices), batch_size):
                batch_indices = indices[i:i+batch_size]
                
                # Simulate log probs (in practice, from model forward pass)
                batch_log_probs_chosen = torch.randn(len(batch_indices), device=device) - 1.5
                batch_log_probs_rejected = torch.randn(len(batch_indices), device=device) - 2.5
                batch_ref_log_probs_chosen = batch_log_probs_chosen + torch.randn(len(batch_indices), device=device) * 0.3
                batch_ref_log_probs_rejected = batch_log_probs_rejected + torch.randn(len(batch_indices), device=device) * 0.3
                
                # Training step
                metrics = self.training_step(
                    batch_log_probs_chosen,
                    batch_log_probs_rejected,
                    batch_ref_log_probs_chosen,
                    batch_ref_log_probs_rejected,
                    optimizer,
                    model
                )
                
                all_metrics.append(metrics)
        
        return all_metrics

# Create a simple dummy model
class DummyModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = torch.nn.Linear(10, 1)
    
    def forward(self, x):
        return self.linear(x)

# Initialize trainer and model
trainer = DPOTrainer(beta=0.5, learning_rate=1e-4, device=device)
model = DummyModel().to(device)

# Dummy preference data (10 examples)
preference_data = [{'prompt': f'q{i}', 'chosen': f'a{i}', 'rejected': f'b{i}'} for i in range(10)]

# Train
metrics = trainer.train_epoch(model, preference_data, batch_size=4, num_epochs=2)

print(f'Trained for {len(metrics)} steps')
print(f'First step - Loss: {metrics[0]["loss"]:.4f}, Accuracy: {metrics[0]["accuracy"]:.3f}')
print(f'Last step - Loss: {metrics[-1]["loss"]:.4f}, Accuracy: {metrics[-1]["accuracy"]:.3f}')

## Real-World Example 1: Preference Data Generation and Validation

Generate synthetic preference pairs and validate their quality.

In [ ]:
class PreferencePairGenerator:
    """
    Generate and validate preference pairs for DPO training.
    """
    
    def __init__(self):
        self.pairs = []
    
    def generate_synthetic_pairs(
        self,
        num_pairs: int = 100
    ) -> List[Dict[str, str]]:
        """
        Generate synthetic preference pairs.
        
        Args:
            num_pairs: Number of pairs to generate
        
        Returns:
            List of preference pairs
        """
        prompts = [
            'What is machine learning?',
            'How does gradient descent work?',
            'Explain neural networks',
            'What is overfitting?',
            'Describe batch normalization'
        ]
        
        pairs = []
        for i in range(num_pairs):
            prompt = prompts[i % len(prompts)]
            # Chosen response: longer, more detailed
            chosen = f'Regarding {prompt}, machine learning is a fundamental approach... with multiple techniques and extensive applications in modern AI systems.'
            # Rejected response: shorter, less detailed
            rejected = f'{prompt}? It is a technique.'
            
            pairs.append({
                'prompt': prompt,
                'chosen': chosen,
                'rejected': rejected
            })
        
        self.pairs = pairs
        return pairs
    
    def compute_preference_quality_metrics(self) -> Dict[str, float]:
        """
        Compute quality metrics for preference pairs.
        
        Returns:
            Dictionary of quality metrics
        """
        if not self.pairs:
            return {}
        
        chosen_lengths = [len(p['chosen'].split()) for p in self.pairs]
        rejected_lengths = [len(p['rejected'].split()) for p in self.pairs]
        
        # Check overlap between chosen and rejected
        overlaps = []
        for pair in self.pairs:
            chosen_words = set(pair['chosen'].lower().split())
            rejected_words = set(pair['rejected'].lower().split())
            if len(chosen_words | rejected_words) > 0:
                overlap = len(chosen_words & rejected_words) / len(chosen_words | rejected_words)
                overlaps.append(overlap)
        
        return {
            'num_pairs': len(self.pairs),
            'avg_chosen_length': float(np.mean(chosen_lengths)),
            'avg_rejected_length': float(np.mean(rejected_lengths)),
            'length_difference': float(np.mean(np.array(chosen_lengths) - np.array(rejected_lengths))),
            'avg_response_overlap': float(np.mean(overlaps)),
        }

# Generate and validate preferences
generator = PreferencePairGenerator()
pairs = generator.generate_synthetic_pairs(num_pairs=50)
metrics = generator.compute_preference_quality_metrics()

print('Preference Data Quality Metrics:')
for key, value in metrics.items():
    if isinstance(value, float):
        print(f'  {key}: {value:.3f}')
    else:
        print(f'  {key}: {value}')

print(f'\nExample pair:')
print(f"  Prompt: {pairs[0]['prompt']}")
print(f"  Chosen (length {len(pairs[0]['chosen'].split())}): {pairs[0]['chosen'][:80]}...")
print(f"  Rejected (length {len(pairs[0]['rejected'].split())}): {pairs[0]['rejected'][:80]}...")

## Real-World Example 2: DPO vs RLHF Efficiency Comparison

Analyze the computational and efficiency differences between DPO and RLHF.

In [ ]:
def compare_dpo_vs_rlhf() -> Dict:
    """
    Compare DPO and RLHF on key efficiency metrics.
    
    Returns:
        Comparison dictionary with metrics
    """
    comparison = {
        'rlhf': {
            'pipeline_stages': 3,  # SFT, Reward Model, RL
            'training_time_hours': 72,
            'memory_usage_gb': 120,
            'reward_model_params': '7B',
            'stability': 'Medium (RL instability)',
            'components': ['SFT Model', 'Reward Model', 'Policy Model (RL)']
        },
        'dpo': {
            'pipeline_stages': 1,  # DPO only
            'training_time_hours': 12,
            'memory_usage_gb': 60,
            'reward_model_params': 'None (implicit)',
            'stability': 'High (supervised learning)',
            'components': ['SFT Model', 'Reference Model (copy)']
        }
    }
    
    # Compute efficiency ratios
    speedup = comparison['rlhf']['training_time_hours'] / comparison['dpo']['training_time_hours']
    memory_ratio = comparison['rlhf']['memory_usage_gb'] / comparison['dpo']['memory_usage_gb']
    
    comparison['efficiency_gains'] = {
        'speedup': f'{speedup:.1f}x',
        'memory_reduction': f'{memory_ratio:.1f}x',
        'fewer_pipeline_stages': comparison['rlhf']['pipeline_stages'] - comparison['dpo']['pipeline_stages']
    }
    
    return comparison

# Run comparison
comparison = compare_dpo_vs_rlhf()

print('=== DPO vs RLHF Comparison ===' )
print('\nRLHF:')
for key, val in comparison['rlhf'].items():
    print(f'  {key}: {val}')

print('\nDPO:')
for key, val in comparison['dpo'].items():
    print(f'  {key}: {val}')

print('\nEfficiency Gains of DPO:')
for key, val in comparison['efficiency_gains'].items():
    print(f'  {key}: {val}')

# Visualize training time
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Time comparison
methods = ['RLHF', 'DPO']
times = [comparison['rlhf']['training_time_hours'], comparison['dpo']['training_time_hours']]
ax1.bar(methods, times, color=['coral', 'lightgreen'])
ax1.set_ylabel('Training Time (hours)')
ax1.set_title('Training Time Comparison')
ax1.set_ylim(0, max(times) * 1.1)
for i, v in enumerate(times):
    ax1.text(i, v + 2, f'{v}h', ha='center', fontweight='bold')

# Memory comparison
memories = [comparison['rlhf']['memory_usage_gb'], comparison['dpo']['memory_usage_gb']]
ax2.bar(methods, memories, color=['coral', 'lightgreen'])
ax2.set_ylabel('Memory Usage (GB)')
ax2.set_title('Memory Usage Comparison')
ax2.set_ylim(0, max(memories) * 1.1)
for i, v in enumerate(memories):
    ax2.text(i, v + 3, f'{v}GB', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('\nKey takeaway: DPO is 6x faster and 2x more memory-efficient than RLHF')

## Key Takeaways

**Core idea:** DPO optimizes language models directly on preference pairs without training a separate reward model.

**Mathematical insight:** The DPO loss derives from rearranging the RLHF objective, showing that reward models aren't necessary—the log probability ratio contains all needed information.

**DPO advantages vs RLHF:**
| Aspect | RLHF | DPO |
|--------|------|-----|
| Training time | 72 hours | 12 hours (6x faster) |
| Memory usage | 120 GB | 60 GB (2x less) |
| Stability | Medium (RL instability) | High (supervised) |
| Components | 3 models | 2 models |
| Interpretability | Learned reward (black box) | Explicit preference ratio |

**When to use each:**
- DPO: Single-objective alignment, limited compute, rapid iteration
- RLHF: Multi-objective optimization, complex preferences, established workflows

**Common pitfalls in DPO:**
- Noisy preference data (directly harms training)
  → Fix: Validate preference quality, filter ambiguous pairs
- Too-high beta (mode collapse, reduced diversity)
  → Fix: Use moderate beta (0.3-0.5), monitor output diversity
- Divergence from reference model (instability)
  → Fix: Limit training iterations, use moderate beta, monitor log ratios

## Exercises

1. **Experiment with beta**: Change beta from 0.1 to 1.0 and observe how preference enforcement changes. What's the trade-off?

2. **Analyze log ratios**: Track model_log_ratio vs ref_log_ratio during training. When does the model diverge too far from the reference?

3. **Create preference data**: Design a system to automatically generate preference pairs (e.g., by scoring with heuristics). What makes good vs bad preferences?

4. **Compare to RLHF**: Implement a simplified RLHF loss and compare convergence speed and stability with DPO. Which is more stable?